# Shared breath offset and centered local correction for v009

Definitions only. Execute `research/sequence_components_v005.ipynb` first in the same namespace. These cells read no assignment data, launch no fitting or remote work, and perform no submission. The next batch is conditional on v008 producing no qualifying candidate, under the standing approval in `research/authorization_v008.json`.

The original compact encoder and scalar head retain their parameter names, shapes and initialization. A width16 MLP predicts one shared offset from concatenated inspiratory mean/max encoder states. A separate width16 MLP predicts local corrections. Prediction is `original_scalar + shared_offset + local - inspiratory_mean(local)`, with the local correction centered within each breath. Both final projections start at zero, so the initialized model equals the common baseline. No extra dropout or auxiliary loss is added. The redundant final local bias is omitted because centering would cancel it exactly: the compact bidirectional128 model adds12,353 trainable parameters.

The mask comes only from the known `u_out` feature, reconstructed with the supplied fitter-only `feature_names`, `x_mean` and `x_std`. Its index and selected scaler statistics are checkpoint buffers. No target statistic or pressure label enters this component. Pooling excludes encoded expiration positions; the unchanged bidirectional encoder intentionally still uses the complete known breath inputs. An all-expiration inference breath gets zero correction and finite baseline predictions.

`v9_offset_model(spec, n_features, preprocessing)` supports the common v008 trainer and its tiny hidden8 proof. `v9_offset_selfcheck(device='cpu')` explicitly checks initialization, real two-step branch gradients, masking/centering, independence/reset, known-input contracts and serialized scaler restoration. The caller owns grouped fitting, masked scalar L1, frozen rates, source hashes, actual-GPU strict-deterministic capacity/resume gates, and promotion decisions. These synthetic CPU checks do not establish Torch2.2 CUDA support or performance.


In [ ]:
# Reuse the unchanged v005 global namespace; definitions only.
_V9_OFFSET_REQUIRED = ('V5PressureRNN', 'canonical_model_spec', '_sequence_device',
    'seed_sequence', '_rng_state', '_restore_rng', 'torch', 'nn', 'np', 'V5_STEPS')
if any(name not in globals() for name in _V9_OFFSET_REQUIRED):
    raise RuntimeError('Execute research/sequence_components_v005.ipynb first')

V9_OFFSET_VERSION = 'v009-shared-offset-centered-local-width16-v1'
V9_OFFSET_WIDTH = 16


def _v9_offset_context(preprocessing, n_features):
    """Read only fitter-only known-input names/scalers, never target statistics."""
    names = preprocessing.get('feature_names')
    if names is None or len(names) != int(n_features) or len(set(names)) != len(names):
        raise ValueError('Offset model requires unique training-preprocessor feature_names')
    names = tuple(names)
    if not all(isinstance(name, str) for name in names):
        raise ValueError('Feature names must be strings')
    if any('pressure' in name.lower() for name in names):
        raise ValueError('Pressure-derived inputs are not allowed')
    if 'u_out' not in names:
        raise ValueError('Known u_out input is required for inspiratory pooling')
    mean = np.asarray(preprocessing['x_mean'], dtype=np.float32)
    std = np.asarray(preprocessing['x_std'], dtype=np.float32)
    if mean.shape != (int(n_features),) or std.shape != mean.shape:
        raise ValueError('Training feature scaler dimension mismatch')
    if not np.isfinite(mean).all() or not np.isfinite(std).all() or (std <= 0).any():
        raise ValueError('Invalid training feature scaler')
    index = names.index('u_out')
    return names, index, float(mean[index]), float(std[index])


class V9OffsetPressureRNN(V5PressureRNN):
    """Original scalar model plus a shared offset and zero-mean local correction.

    Every reduction is within one breath. The pooling mask is a known valve
    input recovered from fitter-only checkpointed scaling. No state persists
    across calls, and no prediction-time target or residual label is accepted.
    """
    def __init__(self, model_spec, n_features, preprocessing):
        spec = canonical_model_spec(model_spec)
        if spec['rc_heads']:
            raise ValueError('v009 offset model preserves the original scalar head')
        super().__init__(spec, n_features)
        baseline_state = self.state_dict()
        names, index, mean, std = _v9_offset_context(preprocessing, n_features)
        self.offset_feature_names = names
        self.register_buffer('offset_u_out_index', torch.tensor([index], dtype=torch.long))
        self.register_buffer('offset_u_out_mean', torch.tensor(mean, dtype=torch.float32))
        self.register_buffer('offset_u_out_std', torch.tensor(std, dtype=torch.float32))
        encoder_width = self.head[0].in_features
        self.offset_global_head = nn.Sequential(nn.Linear(2*encoder_width, V9_OFFSET_WIDTH),
            nn.SiLU(), nn.Linear(V9_OFFSET_WIDTH, 1))
        self.offset_local_head = nn.Sequential(nn.Linear(encoder_width, V9_OFFSET_WIDTH),
            nn.SiLU(), nn.Linear(V9_OFFSET_WIDTH, 1, bias=False))
        nn.init.zeros_(self.offset_global_head[-1].weight)
        nn.init.zeros_(self.offset_global_head[-1].bias)
        nn.init.zeros_(self.offset_local_head[-1].weight)
        # Explicitly copy the original common initialization after branch creation.
        self.load_common_base_state(baseline_state)

    def load_common_base_state(self, state_dict):
        """Strict common-weight transfer; ordinary strict loading handles trained models."""
        expected = self.state_dict()
        common = {name for name in expected if not name.startswith('offset_')}
        if set(state_dict) != common:
            raise ValueError('Common baseline state must contain exactly the original v005 model keys')
        for name in common:
            if not torch.is_tensor(state_dict[name]) or state_dict[name].shape != expected[name].shape:
                raise ValueError('Common baseline tensor shape changed: '+name)
        incompatible = self.load_state_dict(state_dict, strict=False)
        added = {name for name in expected if name.startswith('offset_')}
        if set(incompatible.missing_keys) != added or incompatible.unexpected_keys:
            raise RuntimeError('Only new offset-branch keys may be absent from common initialization')
        return incompatible

    def _validate_offset_input(self, x):
        if x.ndim != 3 or x.shape[1] != V5_STEPS or x.shape[-1] != self.n_features+1:
            raise ValueError('Expected complete known input breaths [B,80,F+1] with internal R/C code')

    def inspiration_mask(self, x):
        self._validate_offset_input(x)
        standardized = x.index_select(-1, self.offset_u_out_index).squeeze(-1)
        raw_u_out = standardized*self.offset_u_out_std + self.offset_u_out_mean
        # Raw inputs are binary; threshold tolerates inverse-scaling roundoff.
        return raw_u_out < .5

    def encode(self, x):
        # Exact original V5PressureRNN.forward encoder, including existing dropout.
        self._validate_offset_input(x)
        joint = x[..., -1].long()
        sequence = x[..., :self.n_features]
        if self.joint_embedding is not None:
            sequence = torch.cat((sequence, self.joint_embedding(joint)), dim=-1)
        if not self.explicit_stack:
            sequence, _ = self.rnn(sequence)
        else:
            outputs = []
            for index, layer in enumerate(self.recurrent_layers):
                previous = sequence
                sequence, _ = layer(previous)
                if self.spec['residual']:
                    sequence = sequence + self.skip_projections[index](previous)
                outputs.append(sequence)
                if index+1 < len(self.recurrent_layers):
                    sequence = self.between_layers(sequence)
            if self.spec['output_concat']:
                sequence = torch.cat(outputs, dim=-1)
        return sequence

    def correction_components(self, encoded, mask):
        """Inspect the branch directly, separate from allowed bidirectional context.

        Pooling and centering exclude expiration positions. Empty inspiration
        has finite zero pooled statistics and contributes no correction.
        """
        if encoded.ndim != 3 or encoded.shape[-1] != self.head[0].in_features:
            raise ValueError('Expected original encoder states [B,T,H]')
        if mask.dtype != torch.bool or mask.shape != encoded.shape[:2]:
            raise ValueError('Expected known boolean inspiration mask [B,T]')
        active = mask.any(dim=1, keepdim=True)
        weights = mask.to(dtype=encoded.dtype)
        counts = weights.sum(dim=1, keepdim=True).clamp_min(1.)
        pooled_mean = (encoded*weights.unsqueeze(-1)).sum(dim=1)/counts
        pooled_max = encoded.masked_fill(~mask.unsqueeze(-1),
            torch.finfo(encoded.dtype).min).amax(dim=1)
        pooled_max = torch.where(active, pooled_max, torch.zeros_like(pooled_max))
        shared_offset = self.offset_global_head(torch.cat((pooled_mean, pooled_max), dim=-1))
        shared_offset = torch.where(active, shared_offset, torch.zeros_like(shared_offset))
        local = self.offset_local_head(encoded).squeeze(-1)
        local_mean = (local*weights).sum(dim=1, keepdim=True)/counts
        centered_local = torch.where(active, local-local_mean, torch.zeros_like(local))
        return dict(offset=shared_offset, local=local, centered_local=centered_local,
            correction=shared_offset+centered_local)

    def forward(self, x):
        encoded = self.encode(x)
        baseline = self.head(encoded).squeeze(-1)
        components = self.correction_components(encoded, self.inspiration_mask(x))
        return baseline + components['correction']


def v9_offset_model(model_spec, n_features, preprocessing):
    """Three-argument factory for the unchanged v008 trainer; no target argument."""
    return V9OffsetPressureRNN(model_spec, n_features, preprocessing)


In [ ]:
def v9_offset_selfcheck(device='cpu'):
    """Explicit synthetic checks only; preserve caller RNG/backend configuration."""
    import inspect
    import io
    selected = _sequence_device(device)
    generator = torch.Generator()
    saved_rng = _rng_state(generator)
    backend = dict(threads=torch.get_num_threads(),
        deterministic=torch.are_deterministic_algorithms_enabled(),
        warn_only=torch.is_deterministic_algorithms_warn_only_enabled(),
        benchmark=torch.backends.cudnn.benchmark,
        cudnn_deterministic=torch.backends.cudnn.deterministic,
        matmul_tf32=torch.backends.cuda.matmul.allow_tf32,
        cudnn_tf32=torch.backends.cudnn.allow_tf32)
    checks = {}
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks[name] = True
    def rejects(name, action):
        try:
            action()
        except (ValueError, TypeError):
            checks[name] = True
        else:
            raise AssertionError(name)
    try:
        seed = 20261102
        names = ['R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt', 'unrelated']
        means = np.array([20., 20., 1., 12., .63, 39.5, .03, -4.], dtype=np.float32)
        scales = np.array([15., 10., .8, 9., .48, 23., .005, 3.], dtype=np.float32)
        prep = dict(feature_names=names, x_mean=means, x_std=scales)
        spec = dict(family='lstm', hidden=128, layers=2, dropout=.15,
            bidirectional=True, head_width=32)
        seed_sequence(seed)
        baseline = V5PressureRNN(spec, len(names)).to(selected)
        seed_sequence(seed)
        model = v9_offset_model(spec, len(names), prep).to(selected)
        common_state = {name: value for name, value in model.state_dict().items()
            if not name.startswith('offset_')}
        check('all_common_initial_parameters_exact', common_state.keys() == baseline.state_dict().keys()
            and all(torch.equal(value, common_state[name])
                for name, value in baseline.state_dict().items()))
        added_parameters = sum(parameter.numel() for name, parameter in model.named_parameters()
            if name.startswith('offset_'))
        check('compact_added_parameters_12353', added_parameters == 12353)
        check('original_scalar_head_preserved', model.head[0].in_features == 256
            and model.head[0].out_features == 32 and model.head[-1].out_features == 1)
        check('no_additional_dropout', [(name, layer.p) for name, layer in model.named_modules()
            if isinstance(layer, nn.Dropout)] == [('between_layers', .15)])
        x = torch.randn(3, V5_STEPS, len(names)+1, device=selected)
        raw_out = (torch.arange(V5_STEPS, device=selected)[None, :]
            >= torch.tensor([29, 31, 33], device=selected)[:, None]).to(torch.float32)
        x[..., names.index('u_out')] = (raw_out-float(means[4]))/float(scales[4])
        x[..., -1] = torch.tensor([0., 4., 8.], device=selected)[:, None]
        mask = raw_out == 0
        model.eval(); baseline.eval()
        with torch.no_grad():
            prediction = model(x)
            check('known_u_out_mask_recovered_exactly', torch.equal(model.inspiration_mask(x), mask))
            check('zero_residual_initial_prediction_exact', torch.equal(prediction, baseline(x)))
            check('finite_scalar_output_shape', prediction.shape == (3, V5_STEPS)
                and torch.isfinite(prediction).all())
        model.train(); baseline.train()
        dropout_cpu = torch.get_rng_state()
        dropout_cuda = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        with torch.no_grad():
            train_baseline = baseline(x)
            torch.set_rng_state(dropout_cpu)
            if dropout_cuda is not None:
                torch.cuda.set_rng_state_all(dropout_cuda)
            train_offset = model(x)
        check('same_dropout_draw_initial_prediction_exact', torch.equal(train_baseline, train_offset))

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.)
        target = .3 + .2*x[..., names.index('u_in')] + .1*torch.sin(x[..., 2])
        branch_parameters = {name: parameter for name, parameter in model.named_parameters()
            if name.startswith('offset_')}
        for update in range(2):
            optimizer.zero_grad(set_to_none=True)
            loss = (model(x)-target).abs().masked_select(mask).mean()
            loss.backward()
            check('finite_all_parameter_gradients_update_'+str(update+1), all(
                parameter.grad is not None and torch.isfinite(parameter.grad).all()
                for parameter in model.parameters()))
            if update == 0:
                check('both_output_projections_learn_on_first_update',
                    model.offset_global_head[-1].weight.grad.abs().sum() > 0
                    and model.offset_local_head[-1].weight.grad.abs().sum() > 0)
                check('upstream_branch_gradients_initially_zero',
                    torch.count_nonzero(model.offset_global_head[0].weight.grad) == 0
                    and torch.count_nonzero(model.offset_local_head[0].weight.grad) == 0)
            else:
                check('both_branches_all_parameters_learn_on_second_update', all(
                    parameter.grad.abs().sum() > 0 for parameter in branch_parameters.values()))
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            encoded = model.encode(x)
            parts = model.correction_components(encoded, mask)
            centered_mean = (parts['centered_local']*mask).sum(dim=1)/mask.sum(dim=1)
            check('local_inspiratory_mean_zero', torch.allclose(centered_mean,
                torch.zeros_like(centered_mean), rtol=0., atol=1e-7))
            check('shared_offset_constant_within_breath', parts['offset'].shape == (3, 1)
                and torch.equal(parts['offset'].expand(-1, V5_STEPS),
                    parts['offset'].repeat(1, V5_STEPS)))
            check('correction_inspiratory_mean_equals_offset', torch.allclose(
                (parts['correction']*mask).sum(dim=1)/mask.sum(dim=1),
                parts['offset'].squeeze(-1), rtol=1e-5, atol=1e-7))
            changed = encoded.clone()
            changed[~mask] += 1000.
            altered_parts = model.correction_components(changed, mask)
            check('expiration_excluded_from_global_offset', torch.equal(
                parts['offset'], altered_parts['offset']))
            check('expiration_excluded_from_local_centering', torch.equal(
                parts['centered_local'][mask], altered_parts['centered_local'][mask]))
            check('expiration_excluded_from_inspiratory_correction', torch.equal(
                parts['correction'][mask], altered_parts['correction'][mask]))
            empty = model.correction_components(encoded, torch.zeros_like(mask))
            check('all_expiration_branch_finite_zero_correction',
                torch.isfinite(empty['correction']).all()
                and torch.count_nonzero(empty['correction']) == 0)
            empty_x = x.clone()
            empty_x[..., names.index('u_out')] = (1.-float(means[4]))/float(scales[4])
            check('all_expiration_full_model_finite', torch.isfinite(model(empty_x)).all())
            # Shape/order stay fixed, avoiding different backend kernels across batches.
            after = model(x)
            changed_x = x.clone()
            changed_x[1, :, :len(names)] += 1.
            isolated = model(changed_x)
            check('other_breath_cannot_change_prediction', torch.equal(after[0], isolated[0])
                and torch.equal(after[2], isolated[2]))
            check('own_known_input_changes_prediction', not torch.equal(after[1], isolated[1]))
            model(changed_x)
            check('repeated_call_has_no_persistent_breath_state', torch.equal(after, model(x)))
            permutation = torch.tensor([2, 0, 1], device=selected)
            check('whole_breath_permutation', torch.allclose(after.index_select(0, permutation),
                model(x.index_select(0, permutation)), rtol=1e-5, atol=1e-6))
            unrelated = x.clone()
            unrelated[..., [0, 1, 2, 3, 5, 6, 7, 8]] += 100.
            check('mask_reads_only_known_u_out', torch.equal(mask, model.inspiration_mask(unrelated)))
            # Real serialization; incorrect constructor scaler stats must be replaced
            # by checkpoint buffers before inference reconstructs the known mask.
            serialized = io.BytesIO()
            torch.save(model.state_dict(), serialized)
            serialized.seek(0)
            restored = torch.load(serialized, map_location=selected, weights_only=False)
            wrong_scalers = dict(prep, x_mean=means+2., x_std=scales*3.)
            clone = v9_offset_model(spec, len(names), wrong_scalers).to(selected)
            clone.load_state_dict(restored, strict=True)
            clone.eval()
            check('serialized_fitted_mask_statistics_restored',
                torch.equal(clone.offset_u_out_mean, model.offset_u_out_mean)
                and torch.equal(clone.offset_u_out_std, model.offset_u_out_std)
                and torch.equal(clone.offset_u_out_index, model.offset_u_out_index)
                and torch.equal(clone.inspiration_mask(x), mask))
            check('serialized_clone_prediction_exact', torch.equal(after, clone(x)))
        check('forward_has_only_known_input_argument',
            tuple(inspect.signature(V9OffsetPressureRNN.forward).parameters) == ('self', 'x'))
        check('factory_has_three_argument_trainer_contract',
            tuple(inspect.signature(v9_offset_model).parameters)
                == ('model_spec', 'n_features', 'preprocessing'))
        rejects('reject_pressure_label_argument', lambda: model(x, target))
        rejects('reject_pressure_feature_name', lambda: v9_offset_model(spec, len(names),
            dict(prep, feature_names=['pressure']+names[1:])))
        rejects('reject_missing_u_out_name', lambda: v9_offset_model(spec, len(names),
            dict(prep, feature_names=[name if name != 'u_out' else 'unknown' for name in names])))
        rejects('reject_categorical_rc_heads', lambda: v9_offset_model(dict(spec, rc_heads=True),
            len(names), prep))
        rejects('reject_incomplete_breath', lambda: model(x[:, :79]))
        check('strict_determinism_stays_enabled', torch.are_deterministic_algorithms_enabled()
            and not torch.is_deterministic_algorithms_warn_only_enabled())
        return dict(component=V9_OFFSET_VERSION, synthetic_only=True, dataset_reads=0,
            device=str(selected), torch_version=str(torch.__version__),
            cuda_version=torch.version.cuda, strict_deterministic=True,
            model_spec=spec, parameter_count=sum(p.numel() for p in model.parameters()),
            added_parameters=added_parameters, checks=checks, checks_total=len(checks),
            passed=all(checks.values()), gpu_profile_or_engine_resume_proof=False)
    finally:
        _restore_rng(saved_rng, generator)
        torch.set_num_threads(backend['threads'])
        torch.use_deterministic_algorithms(backend['deterministic'], warn_only=backend['warn_only'])
        torch.backends.cudnn.benchmark = backend['benchmark']
        torch.backends.cudnn.deterministic = backend['cudnn_deterministic']
        torch.backends.cuda.matmul.allow_tf32 = backend['matmul_tf32']
        torch.backends.cudnn.allow_tf32 = backend['cudnn_tf32']
